# 06 — Feature Selection + LightGBM Classifier

**Pipeline:**
1. Build 60+ features from raw OHLCV
2. Train a Random Forest for feature importance (fast proxy)
3. Remove highly correlated features (Pearson > 0.95)
4. Keep top 20 by importance
5. Train LightGBM on selected features
6. Evaluate calibration and directional accuracy
7. Backtest with confidence-gated trading
8. Compare to buy-and-hold

In [1]:
SYMBOL    = 'BTCUSDT'
INTERVAL  = '1h'

TRAIN_END = '2024-06-01'
VAL_END   = '2024-11-10'
# Test: 2024-11-10 → present

TOP_N_FEATURES    = 20
CORR_THRESHOLD    = 0.95   # drop one of any pair with |r| > this
RF_N_ESTIMATORS   = 300    # random forest for importance

# LightGBM
LGB_PARAMS = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'n_estimators':     1000,
    'learning_rate':    0.02,
    'num_leaves':       31,
    'max_depth':        -1,
    'min_child_samples': 50,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'n_jobs':           -1,
    'verbose':          -1,
}
EARLY_STOPPING_ROUNDS = 50

# Trading
CONFIDENCE_THRESHOLD = 0.60
EXIT_THRESHOLD       = 0.45
MIN_HOLD_CANDLES     = 12
MAX_HOLD_CANDLES     = 72
COOLDOWN_CANDLES     = 6
TAKE_PROFIT          = 0.025
STOP_LOSS            = 0.015
FEE                  = 0.0005


In [2]:
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  

import lightgbm as lgb
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

from hmats.data.binance_store import load
from hmats.data.splits import calendar_split

warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'legend.framealpha': 0.85,
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'
RED='#EF5350'; GREEN='#26A69A'; PURPLE='#7B1FA2'

REPO_ROOT = Path.cwd().parents[2]
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = Path.cwd()
RAW_DIR     = REPO_ROOT / 'data' / 'raw'
MODELS_DIR  = REPO_ROOT / 'models'
FIGURES_DIR = REPO_ROOT / 'figures'
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


## 1. Load raw data

In [3]:
raw = load(SYMBOL, INTERVAL, store_dir=str(RAW_DIR))
raw.index = raw.index.tz_localize(None) if raw.index.tz else raw.index
print(f'{len(raw):,} rows  {raw.index.min().date()} → {raw.index.max().date()}')
raw.head(3)


76,237 rows  2017-08-17 → 2026-05-04


,open,high,low,close,volume
open_time,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691


## 2. Feature engineering — 60+ features

Built entirely from OHLCV. No external data required.

In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o = df.copy()
    c, h, l, v = o['close'], o['high'], o['low'], o['volume']

    # ── Returns ───────────────────────────────────────────────────────────────
    for p in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
        o[f'ret_{p}h'] = c.pct_change(p)
        o[f'log_ret_{p}h'] = np.log(c / c.shift(p))

    # ── Volatility ────────────────────────────────────────────────────────────
    log_ret1 = np.log(c / c.shift(1))
    for w in [6, 12, 24, 48, 72, 168]:
        o[f'vol_{w}h'] = log_ret1.rolling(w).std()

    # ── ATR ───────────────────────────────────────────────────────────────────
    tr = pd.concat([
        h - l,
        (h - c.shift(1)).abs(),
        (l - c.shift(1)).abs(),
    ], axis=1).max(axis=1)
    for w in [14, 24]:
        o[f'atr_{w}'] = tr.rolling(w).mean()
        o[f'atr_{w}_pct'] = o[f'atr_{w}'] / c

    # ── Moving averages & ratios ───────────────────────────────────────────────
    for w in [7, 14, 20, 50, 100, 200]:
        sma = c.rolling(w).mean()
        ema = c.ewm(span=w, adjust=False).mean()
        o[f'sma_{w}'] = sma
        o[f'ema_{w}'] = ema
        o[f'close_vs_sma_{w}'] = c / sma - 1
        o[f'close_vs_ema_{w}'] = c / ema - 1

    # ── Bollinger Bands ───────────────────────────────────────────────────────
    for w in [20, 50]:
        mid = c.rolling(w).mean()
        std = c.rolling(w).std()
        o[f'bb_width_{w}']    = (2 * std) / (mid + 1e-12)
        o[f'bb_position_{w}'] = (c - (mid - 2*std)) / (4 * std + 1e-12)

    # ── MACD family ───────────────────────────────────────────────────────────
    for fast, slow, sig in [(12,26,9), (5,13,4)]:
        ema_f = c.ewm(span=fast, adjust=False).mean()
        ema_s = c.ewm(span=slow, adjust=False).mean()
        macd  = ema_f - ema_s
        signal = macd.ewm(span=sig, adjust=False).mean()
        o[f'macd_{fast}_{slow}']      = macd / (c + 1e-12)
        o[f'macd_hist_{fast}_{slow}'] = (macd - signal) / (c + 1e-12)

    # ── RSI variants ──────────────────────────────────────────────────────────
    def rsi(series, period):
        d = series.diff()
        gain = d.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
        loss = (-d.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
        return 100 - 100 / (1 + gain / (loss + 1e-12))
    for p in [7, 14, 21]:
        o[f'rsi_{p}'] = rsi(c, p) / 100   # normalise to [0,1]

    # ── Stochastic ────────────────────────────────────────────────────────────
    for w in [14, 21]:
        lo  = l.rolling(w).min()
        hi  = h.rolling(w).max()
        o[f'stoch_k_{w}'] = (c - lo) / (hi - lo + 1e-12)

    # ── Williams %R ───────────────────────────────────────────────────────────
    hi14 = h.rolling(14).max()
    lo14 = l.rolling(14).min()
    o['williams_r'] = (hi14 - c) / (hi14 - lo14 + 1e-12)

    # ── Volume features ───────────────────────────────────────────────────────
    for w in [12, 24, 72, 168]:
        o[f'vol_z_{w}h'] = (v - v.rolling(w).mean()) / (v.rolling(w).std() + 1e-12)
        o[f'vol_ratio_{w}h'] = v / (v.rolling(w).mean() + 1e-12)

    # OBV
    obv = (np.sign(log_ret1) * v).cumsum()
    o['obv_z_72'] = (obv - obv.rolling(72).mean()) / (obv.rolling(72).std() + 1e-12)

    # ── Candle structure ──────────────────────────────────────────────────────
    o['candle_body']   = (c - o['open']).abs() / (h - l + 1e-12)
    o['upper_wick']    = (h - pd.concat([c, o['open']], axis=1).max(axis=1)) / (h - l + 1e-12)
    o['lower_wick']    = (pd.concat([c, o['open']], axis=1).min(axis=1) - l) / (h - l + 1e-12)
    o['is_bullish']    = (c > o['open']).astype(float)

    # ── High/low position ─────────────────────────────────────────────────────
    for w in [24, 48, 168]:
        hi_w = h.rolling(w).max()
        lo_w = l.rolling(w).min()
        o[f'hl_position_{w}h'] = (c - lo_w) / (hi_w - lo_w + 1e-12)

    # ── Calendar / seasonality ────────────────────────────────────────────────
    o['hour_sin'] = np.sin(2 * np.pi * o.index.hour / 24)
    o['hour_cos'] = np.cos(2 * np.pi * o.index.hour / 24)
    o['dow_sin']  = np.sin(2 * np.pi * o.index.dayofweek / 7)
    o['dow_cos']  = np.cos(2 * np.pi * o.index.dayofweek / 7)

    # ── Label ─────────────────────────────────────────────────────────────────
    o['label'] = (c.shift(-1) > c).astype(int)

    # Drop raw OHLCV and SMAs (too correlated, keeping ratios instead)
    drop_cols = ['open','high','low','close','volume',
                 *[f'sma_{w}' for w in [7,14,20,50,100,200]],
                 *[f'ema_{w}' for w in [7,14,20,50,100,200]]]
    o = o.drop(columns=[c for c in drop_cols if c in o.columns])

    return o.replace([np.inf, -np.inf], np.nan).dropna()


feat_df = build_features(raw)
feature_cols = [c for c in feat_df.columns if c != 'label']
print(f'Total features built: {len(feature_cols)}')
print(f'Rows after dropna:    {len(feat_df):,}')
feat_df.head(3)


Total features built: 74
Rows after dropna:    76,038


,ret_1h,log_ret_1h,ret_2h,log_ret_2h,ret_3h,log_ret_3h,ret_6h,log_ret_6h,ret_12h,log_ret_12h,...,lower_wick,is_bullish,hl_position_24h,hl_position_48h,hl_position_168h,hour_sin,hour_cos,dow_sin,dow_cos,label
open_time,,,,,,,,,,,,,,,,,,,,,
2017-08-25 11:00:00,0.010880,0.010821,0.003846,0.003839,0.006717,0.006695,0.018741,0.018567,0.010487,0.010432,...,0.386703,1.0,0.955768,0.965516,0.989287,2.588190e-01,-0.965926,-0.433884,-0.900969,0
2017-08-25 12:00:00,-0.011827,-0.011897,-0.001075,-0.001076,-0.008026,-0.008059,0.001620,0.001619,-0.012897,-0.012981,...,0.484921,0.0,0.721893,0.794654,0.936203,1.224647e-16,-1.000000,-0.433884,-0.900969,1
2017-08-25 13:00:00,0.000536,0.000536,-0.011297,-0.011361,-0.000540,-0.000540,-0.003927,-0.003935,-0.004849,-0.004861,...,0.296367,1.0,0.721043,0.802306,0.938581,-2.588190e-01,-0.965926,-0.433884,-0.900969,0


## 3. Train / val / test split

In [5]:
train_df, val_df, test_df = calendar_split(feat_df, train_end=TRAIN_END, val_end=VAL_END)

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_val   = val_df[feature_cols].values
y_val   = val_df['label'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['label'].values

print(f'Train: {len(train_df):>7,}  {train_df.index.min().date()} → {train_df.index.max().date()}  label={y_train.mean():.3f}')
print(f'Val:   {len(val_df):>7,}  {val_df.index.min().date()} → {val_df.index.max().date()}  label={y_val.mean():.3f}')
print(f'Test:  {len(test_df):>7,}  {test_df.index.min().date()} → {test_df.index.max().date()}  label={y_test.mean():.3f}')


Train:  59,190  2017-08-25 → 2024-06-01  label=0.510
Val:     3,888  2024-06-01 → 2024-11-10  label=0.505
Test:   12,960  2024-11-10 → 2026-05-04  label=0.504


## 4. Random Forest — feature importance

Fast proxy model trained on the full feature set.
Used only to rank features — not for trading decisions.

In [6]:
rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=8,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature':   feature_cols,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(f'RF val accuracy: {(rf.predict(X_val) == y_val).mean():.4f}')
importance_df.head(30)


RF val accuracy: 0.5388


,feature,importance
0,log_ret_2h,0.051081
1,ret_2h,0.044726
2,close_vs_sma_7,0.040580
3,close_vs_ema_7,0.038096
4,ret_1h,0.035987
5,log_ret_1h,0.035639
6,log_ret_3h,0.028964
7,ret_3h,0.027483
8,upper_wick,0.025896
9,lower_wick,0.024806


### Feature importance plot (top 40)

In [7]:
top40 = importance_df.head(40)
fig, ax = plt.subplots(figsize=(8, 10))
bars = ax.barh(top40['feature'][::-1], top40['importance'][::-1],
               color=BLUE, alpha=0.75)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Random Forest — Feature Importance (Top 40)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'feature_importance_rf.png')
plt.show()


## 5. Correlation filtering

Among features with high mutual correlation (|r| > 0.95),
keep only the one with higher importance. Then take the top 20.

In [8]:
# Compute correlation matrix on train set
corr_matrix = pd.DataFrame(X_train, columns=feature_cols).corr().abs()

# Greedy removal: iterate by importance rank, drop correlated followers
ranked = importance_df['feature'].tolist()
kept   = []
dropped_corr = []

for feat in ranked:
    if feat not in kept:
        # Check against already-kept features
        too_correlated = False
        for k in kept:
            if corr_matrix.loc[feat, k] > CORR_THRESHOLD:
                too_correlated = True
                dropped_corr.append((feat, k, corr_matrix.loc[feat, k]))
                break
        if not too_correlated:
            kept.append(feat)

print(f'Features after correlation filter: {len(kept)} (dropped {len(dropped_corr)})')
print(f'\nDropped due to high correlation (>{CORR_THRESHOLD}):')
for feat, kept_feat, corr_val in dropped_corr[:15]:
    print(f'  {feat:<35} corr={corr_val:.3f} with {kept_feat}')


Features after correlation filter: 51 (dropped 23)

Dropped due to high correlation (>0.95):
  ret_2h                              corr=1.000 with log_ret_2h
  close_vs_ema_7                      corr=0.980 with close_vs_sma_7
  log_ret_1h                          corr=1.000 with ret_1h
  ret_3h                              corr=0.999 with log_ret_3h
  stoch_k_14                          corr=1.000 with williams_r
  ret_6h                              corr=0.999 with log_ret_6h
  hl_position_24h                     corr=0.973 with stoch_k_21
  atr_14                              corr=0.977 with atr_24
  close_vs_sma_14                     corr=0.982 with close_vs_ema_14
  vol_z_168h                          corr=0.951 with vol_ratio_168h
  ret_168h                            corr=0.994 with log_ret_168h
  close_vs_ema_20                     corr=0.984 with close_vs_ema_14
  close_vs_sma_20                     corr=0.974 with close_vs_ema_14
  log_ret_12h                         corr=0.

In [9]:
selected_features = kept[:TOP_N_FEATURES]
print(f'\nFinal selected features ({len(selected_features)}):')
for i, f in enumerate(selected_features, 1):
    imp = importance_df[importance_df['feature'] == f]['importance'].values[0]
    print(f'  {i:>2}. {f:<40} importance={imp:.4f}')



Final selected features (20):
   1. log_ret_2h                               importance=0.0511
   2. close_vs_sma_7                           importance=0.0406
   3. ret_1h                                   importance=0.0360
   4. log_ret_3h                               importance=0.0290
   5. upper_wick                               importance=0.0259
   6. lower_wick                               importance=0.0248
   7. williams_r                               importance=0.0224
   8. rsi_7                                    importance=0.0215
   9. log_ret_6h                               importance=0.0184
  10. atr_24                                   importance=0.0158
  11. stoch_k_21                               importance=0.0157
  12. macd_hist_5_13                           importance=0.0156
  13. candle_body                              importance=0.0136
  14. is_bullish                               importance=0.0135
  15. close_vs_ema_14                          importance=0

## 6. LightGBM — train on selected features

In [10]:
feat_idx = [feature_cols.index(f) for f in selected_features]

Xtr = X_train[:, feat_idx]
Xvl = X_val[:,   feat_idx]
Xte = X_test[:,  feat_idx]

lgb_train = lgb.Dataset(Xtr, label=y_train, feature_name=selected_features)
lgb_val   = lgb.Dataset(Xvl, label=y_val,   feature_name=selected_features, reference=lgb_train)

callbacks = [
    lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
    lgb.log_evaluation(period=50),
]

lgb_model = lgb.train(
    LGB_PARAMS,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)

lgb_model.save_model(str(MODELS_DIR / 'lgbm_model.txt'))
pd.Series(selected_features).to_csv(MODELS_DIR / 'lgbm_features.csv', index=False, header=False)
print(f'Best iteration: {lgb_model.best_iteration}')


[50]	train's binary_logloss: 0.6806	val's binary_logloss: 0.688627
[100]	train's binary_logloss: 0.674659	val's binary_logloss: 0.688151
[150]	train's binary_logloss: 0.67039	val's binary_logloss: 0.688487
Best iteration: 108


## 7. Evaluation on val and test sets

In [11]:
probs_val  = lgb_model.predict(Xvl)
probs_test = lgb_model.predict(Xte)

val_acc  = ((probs_val  > 0.5) == y_val).mean()
test_acc = ((probs_test > 0.5) == y_test).mean()
val_auc  = roc_auc_score(y_val,  probs_val)
test_auc = roc_auc_score(y_test, probs_test)

print(f'Val  — accuracy: {val_acc:.4f}   AUC: {val_auc:.4f}')
print(f'Test — accuracy: {test_acc:.4f}   AUC: {test_auc:.4f}')
print()
print('Test classification report:')
print(classification_report(y_test, (probs_test > 0.5).astype(int), target_names=['down','up']))


Val  — accuracy: 0.5383   AUC: 0.5576
Test — accuracy: 0.5283   AUC: 0.5476

Test classification report:
              precision    recall  f1-score   support

        down       0.53      0.47      0.50      6422
          up       0.53      0.59      0.56      6538

    accuracy                           0.53     12960
   macro avg       0.53      0.53      0.53     12960
weighted avg       0.53      0.53      0.53     12960



### Confidence distribution

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, probs, split, y in [
    (axes[0], probs_val,  'Validation', y_val),
    (axes[1], probs_test, 'Test',       y_test),
]:
    ax.hist(probs[y == 0], bins=50, alpha=0.6, color=RED,   label='True: down', density=True)
    ax.hist(probs[y == 1], bins=50, alpha=0.6, color=GREEN, label='True: up',   density=True)
    ax.axvline(0.5, color=GREY, ls='--', lw=1)
    ax.axvline(CONFIDENCE_THRESHOLD, color=BLUE, ls='--', lw=1.2,
               label=f'Entry threshold ({CONFIDENCE_THRESHOLD})')
    ax.set_xlabel('P(next candle up)')
    ax.set_ylabel('Density')
    ax.set_title(f'{split} — Confidence Distribution')
    ax.legend(fontsize=8)

fig.suptitle('LightGBM — Predicted Probability Distribution', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'lgbm_confidence_dist.png')
plt.show()


### Calibration curve

Does P=0.65 actually mean 65% of those predictions are correct?
A well-calibrated model follows the diagonal.

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, probs, split, y in [
    (axes[0], probs_val,  'Validation', y_val),
    (axes[1], probs_test, 'Test',       y_test),
]:
    frac_pos, mean_pred = calibration_curve(y, probs, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color=ACCENT, lw=1.5, label='LightGBM')
    ax.plot([0,1],[0,1], ls='--', color=GREY, lw=1, label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'{split} — Calibration Curve')
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('LightGBM Calibration — Does P(up)=X mean X% correct?', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'lgbm_calibration.png')
plt.show()


### Accuracy by confidence bucket

What is the actual win rate when the model is highly confident?

In [14]:
buckets = np.arange(0.5, 1.0, 0.05)
rows = []
for lo in buckets:
    hi   = lo + 0.05
    mask = (probs_test >= lo) & (probs_test < hi)
    n    = mask.sum()
    if n < 10:
        continue
    acc  = (y_test[mask] == 1).mean()
    rows.append({'bucket': f'{lo:.2f}–{hi:.2f}', 'n': n, 'win_rate': acc})

bucket_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [GREEN if r > 0.5 else RED for r in bucket_df['win_rate']]
ax.bar(bucket_df['bucket'], bucket_df['win_rate'], color=colors, alpha=0.8)
ax.axhline(0.5, color=GREY, ls='--', lw=1, label='Random (50%)')
ax.axvline(bucket_df[bucket_df['bucket'].str.startswith(f'{CONFIDENCE_THRESHOLD:.2f}')].index[0] - 0.5
           if not bucket_df[bucket_df['bucket'].str.startswith(f'{CONFIDENCE_THRESHOLD:.2f}')].empty else -1,
           color=BLUE, ls=':', lw=1.2)
for i, row in bucket_df.iterrows():
    ax.text(i, row['win_rate'] + 0.005, f"n={row['n']}", ha='center', fontsize=7.5)
ax.set_xlabel('Confidence bucket'); ax.set_ylabel('Win rate (fraction up)')
ax.set_title('Win Rate by Confidence Bucket — Test Set', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'lgbm_win_rate_by_confidence.png')
plt.show()

print(bucket_df.to_string(index=False))


   bucket    n  win_rate
0.50–0.55 4415  0.501699
0.55–0.60 2171  0.569323
0.60–0.65  590  0.581356
0.65–0.70   65  0.600000


## 8. Backtest

Entry: `P(up) ≥ CONFIDENCE_THRESHOLD` and flat and not in cooldown.

Exit priority:
1. Stop-loss (−1.5%)
2. Take-profit (+2.5%)
3. Max hold (72h) — force exit
4. Confidence < EXIT_THRESHOLD **and** held ≥ MIN_HOLD_CANDLES

In [15]:
close_prices = test_df['close'].values if 'close' in test_df.columns else raw.loc[test_df.index, 'close'].values
# re-attach close price to test_df for backtest
test_close = raw.loc[test_df.index, 'close'].values
signal_index = test_df.index

cash = 1.0; units = 0.0
in_pos = False; entry_px = 0.0; entry_ts = None
hold_count = 0; cooldown = 0

equity_curve = [1.0]
trade_log    = []

for i, (ts, px, conf) in enumerate(zip(signal_index, test_close, probs_test)):
    if cooldown > 0:
        cooldown -= 1

    if in_pos:
        hold_count += 1
        pnl = (px - entry_px) / entry_px
        reason = None
        if pnl <= -STOP_LOSS:                                      reason = 'sl'
        elif pnl >= TAKE_PROFIT:                                   reason = 'tp'
        elif hold_count >= MAX_HOLD_CANDLES:                       reason = 'max_hold'
        elif hold_count >= MIN_HOLD_CANDLES and conf < EXIT_THRESHOLD: reason = 'conf'

        if reason:
            cash = units * px * (1 - FEE)
            trade_log.append({'entry_time': entry_ts, 'exit_time': ts,
                               'entry_px': entry_px,  'exit_px': px,
                               'pnl_pct': pnl,        'reason': reason,
                               'hold_h': hold_count})
            units = 0.0; in_pos = False; hold_count = 0
            cooldown = COOLDOWN_CANDLES

    if not in_pos and cooldown == 0 and conf >= CONFIDENCE_THRESHOLD:
        units = cash * (1 - FEE) / px
        cash  = 0.0; in_pos = True
        entry_px = px; entry_ts = ts; hold_count = 0

    equity_curve.append(cash + units * px)

if in_pos:
    px = test_close[-1]
    cash = units * px * (1 - FEE)
    trade_log.append({'entry_time': entry_ts, 'exit_time': signal_index[-1],
                       'entry_px': entry_px,  'exit_px': px,
                       'pnl_pct': (px-entry_px)/entry_px, 'reason': 'eod', 'hold_h': hold_count})
    equity_curve[-1] = cash

equity_arr = np.array(equity_curve[1:])
trades_df  = pd.DataFrame(trade_log)
print(f'Total trades: {len(trades_df)}')
if not trades_df.empty:
    print(trades_df['reason'].value_counts().to_string())
    print(f'Avg hold: {trades_df["hold_h"].mean():.1f}h   Median: {trades_df["hold_h"].median():.1f}h')


Total trades: 272
reason
conf    147
sl       97
tp       28
Avg hold: 13.5h   Median: 13.0h


## 9. Buy-and-hold benchmark

In [16]:
bh_units  = (1 - FEE) / test_close[0]
bh_equity = bh_units * test_close
bh_final  = bh_equity[-1] * (1 - FEE)


## 10. Metrics

In [17]:
def sharpe(eq, ann=24*365):
    r = np.log(eq[1:]/(eq[:-1]+1e-12))
    return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(ann)) if len(r)>1 else 0.0

def max_dd(eq):
    pk = np.maximum.accumulate(eq)
    return float(((eq-pk)/(pk+1e-12)).min())

metrics = pd.DataFrame([
    {'Strategy': 'LightGBM Agent',
     'Total Return':  f'{(equity_arr[-1]-1)*100:.2f}%',
     'Sharpe (ann.)': f'{sharpe(equity_arr):.3f}',
     'Max Drawdown':  f'{max_dd(equity_arr)*100:.2f}%',
     'Num Trades':    len(trades_df),
     'Win Rate':      f'{(trades_df["pnl_pct"]>0).mean()*100:.1f}%' if not trades_df.empty else 'N/A',
     'Avg Hold (h)':  f'{trades_df["hold_h"].mean():.1f}' if not trades_df.empty else 'N/A',
    },
    {'Strategy': 'Buy & Hold',
     'Total Return':  f'{(bh_final-1)*100:.2f}%',
     'Sharpe (ann.)': f'{sharpe(bh_equity):.3f}',
     'Max Drawdown':  f'{max_dd(bh_equity)*100:.2f}%',
     'Num Trades': 1, 'Win Rate': 'N/A', 'Avg Hold (h)': 'N/A',
    },
]).set_index('Strategy')
print(metrics.to_string())


               Total Return Sharpe (ann.) Max Drawdown  Num Trades Win Rate Avg Hold (h)
Strategy                                                                                
LightGBM Agent      -39.12%        -1.377      -51.92%         272    52.2%         13.5
Buy & Hold            2.30%         0.035      -50.08%           1      N/A          N/A


## 11. Results plot

In [18]:
fig, axes = plt.subplots(3, 1, figsize=(13, 12),
                         gridspec_kw={'height_ratios':[3,1.2,1.2],'hspace':0.10})

# Equity
ax = axes[0]
ax.plot(signal_index, equity_arr, color=ACCENT, lw=1.4, label='LightGBM Agent')
ax.plot(signal_index, bh_equity,  color=BLUE,   lw=1.2, ls='--', label='Buy & Hold')
ax.axhline(1.0, color=GREY, lw=0.7, ls=':')
if not trades_df.empty:
    eq_s = pd.Series(equity_arr, index=signal_index)
    for _, row in trades_df.iterrows():
        ei = eq_s.index.get_indexer([row['entry_time']], method='nearest')[0]
        xi = eq_s.index.get_indexer([row['exit_time']],  method='nearest')[0]
        ax.scatter(eq_s.index[ei], equity_arr[ei], marker='^', color=GREEN, s=30, zorder=5)
        ax.scatter(eq_s.index[xi], equity_arr[xi], marker='v', color=RED,   s=30, zorder=5)
    from matplotlib.lines import Line2D
    handles = list(ax.get_legend_handles_labels()[0]) + [
        Line2D([0],[0],marker='^',color='w',markerfacecolor=GREEN,markersize=8,label='Entry'),
        Line2D([0],[0],marker='v',color='w',markerfacecolor=RED,  markersize=8,label='Exit'),
    ]
    ax.legend(handles=handles, ncol=2)
ax.set_ylabel('Portfolio Value (start=1.0)')
ax.set_title('LightGBM Agent vs Buy & Hold — Test Period (2024-11-10 →)', fontweight='bold')
ax.grid(axis='y',alpha=0.3); ax.grid(axis='x',alpha=0.15)

# Confidence
ax = axes[1]
ax.plot(signal_index, probs_test, color=BLUE, lw=0.6, alpha=0.6, label='P(up)')
ax.axhline(CONFIDENCE_THRESHOLD, color=GREEN, ls='--', lw=1.1, label=f'Entry ≥{CONFIDENCE_THRESHOLD}')
ax.axhline(EXIT_THRESHOLD,       color=RED,   ls='--', lw=1.1, label=f'Exit <{EXIT_THRESHOLD}')
ax.axhline(0.5, color=GREY, ls=':', lw=0.7)
ax.set_ylim(0,1); ax.set_ylabel('P(next candle up)')
ax.set_title('Model Confidence'); ax.legend(ncol=3)
ax.grid(axis='y',alpha=0.3); ax.grid(axis='x',alpha=0.15)

# Drawdown
ax = axes[2]
pk_l = np.maximum.accumulate(equity_arr)
pk_b = np.maximum.accumulate(bh_equity)
ax.fill_between(signal_index, (equity_arr-pk_l)/(pk_l+1e-12)*100, 0, color=ACCENT, alpha=0.45, label='LightGBM')
ax.fill_between(signal_index, (bh_equity -pk_b)/(pk_b +1e-12)*100, 0, color=BLUE,   alpha=0.25, label='Buy & Hold')
ax.set_ylabel('Drawdown (%)'); ax.set_title('Underwater Equity')
ax.legend(); ax.grid(axis='y',alpha=0.3); ax.grid(axis='x',alpha=0.15)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.text(0.5, 0.005,
    f'LightGBM  |  features={TOP_N_FEATURES}, corr_thresh={CORR_THRESHOLD}  |  '
    f'entry≥{CONFIDENCE_THRESHOLD}, exit<{EXIT_THRESHOLD} after {MIN_HOLD_CANDLES}h, '
    f'max_hold={MAX_HOLD_CANDLES}h, cooldown={COOLDOWN_CANDLES}h  |  '
    f'TP={TAKE_PROFIT*100:.1f}%, SL={STOP_LOSS*100:.1f}%',
    ha='center', fontsize=7.5, color=GREY, style='italic')

fig.savefig(FIGURES_DIR / 'lgbm_backtest_results.png')
plt.show()


## 12. Trade log

In [19]:
if not trades_df.empty:
    trades_df['pnl_fmt'] = trades_df['pnl_pct'].apply(lambda x: f'{x*100:+.2f}%')
    display(trades_df[['entry_time','exit_time','hold_h','entry_px','exit_px','pnl_fmt','reason']])


,entry_time,exit_time,hold_h,entry_px,exit_px,pnl_fmt,reason
0,2024-11-13 05:00:00,2024-11-13 13:00:00,8,86545.99,89152.01,+3.01%,tp
1,2024-11-14 06:00:00,2024-11-14 15:00:00,9,89300.69,87947.99,-1.51%,sl
2,2024-11-17 18:00:00,2024-11-18 15:00:00,21,89936.00,92309.51,+2.64%,tp
3,2024-11-22 14:00:00,2024-11-23 14:00:00,24,97296.00,98744.37,+1.49%,conf
4,2024-11-24 11:00:00,2024-11-25 03:00:00,16,97280.03,97898.80,+0.64%,conf
...,...,...,...,...,...,...,...
267,2026-04-21 19:00:00,2026-04-22 02:00:00,7,75023.43,77604.71,+3.44%,tp
268,2026-04-24 21:00:00,2026-04-25 23:00:00,26,77536.48,77625.00,+0.11%,conf
269,2026-04-27 16:00:00,2026-04-28 20:00:00,28,76618.59,76463.95,-0.20%,conf
270,2026-04-29 16:00:00,2026-04-30 07:00:00,15,75803.54,76161.67,+0.47%,conf
